# LeetCode 1509 — Minimum Difference Between Largest and Smallest Value in Three Moves bằng Python

Link đề: https://leetcode.com/problems/minimum-difference-between-largest-and-smallest-value-in-three-moves/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def minDifference(self, nums: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra phép biến đổi then chốt: **"đổi một phần tử thành giá trị tuỳ ý" ⇔ "xoá phần tử đó"** khi chỉ quan tâm `max − min`.
2. Chứng minh vì sao các phần tử nên xoá luôn là **một số phần tử nhỏ nhất và một số phần tử lớn nhất** → chỉ có 4 phương án.
3. Viết được 3 cách: brute force thử mọi bộ ≤ 3 phần tử, **sắp xếp + thử 4 phương án** `O(n log n)`, `heapq.nsmallest/nlargest` `O(n)`; và tổng quát hoá cho `k` bước bằng cửa sổ trượt.
4. Xử lý edge case `n ≤ 4` (đáp án luôn 0) và tránh bẫy **chỉ số âm** trong Python.
5. Thấy được vì sao các tham lam "cục bộ" (xoá đầu có khoảng cách lớn hơn) có thể sai.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Mỗi **bước**, bạn chọn một phần tử và đổi nó thành **giá trị bất kỳ**. Trả về **hiệu nhỏ nhất** giữa giá trị lớn nhất và nhỏ nhất của `nums` sau **tối đa 3 bước**.

**Ví dụ 1:** `nums = [5,3,2,4]` → `0` (đổi 2, 4, 5 thành 3).

**Ví dụ 2:** `nums = [1,5,0,10,14]` → `1` (đổi 5 → 0, 10 → 0, 14 → 1; mảng `[1,0,0,0,1]`). Không thể đạt 0.

**Ví dụ 3:** `nums = [3,100,20]` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `−10⁹ ≤ nums[i] ≤ 10⁹`.

In [1]:
import heapq
import random
import time
from itertools import combinations
from typing import List

examples = [
    ([5, 3, 2, 4], 0),
    ([1, 5, 0, 10, 14], 1),
    ([3, 100, 20], 0),
    ([6, 6, 0, 1, 1, 4, 6], 2),      # ví dụ quen thuộc trong các phiên bản đề
]
for nums, want in examples:
    print(nums, "->", want)

[5, 3, 2, 4] -> 0
[1, 5, 0, 10, 14] -> 1
[3, 100, 20] -> 0
[6, 6, 0, 1, 1, 4, 6] -> 2


## 3. Kiến thức nền

### 3.1 `sorted` và `list.sort`

`sorted(a)` trả list mới, `a.sort()` sắp xếp tại chỗ. Cả hai `O(n log n)`. Sau khi sắp xếp, `a[0]` nhỏ nhất, `a[-1]` lớn nhất.

In [2]:
a = [1, 5, 0, 10, 14]
b = sorted(a)
print("a =", a, "| sorted =", b)
a.sort()
print("a sau sort =", a, "| min =", a[0], "| max =", a[-1])

a = [1, 5, 0, 10, 14] | sorted = [0, 1, 5, 10, 14]
a sau sort = [0, 1, 5, 10, 14] | min = 0 | max = 14


### 3.2 Chỉ số âm trong Python

`a[-1]` là phần tử cuối, `a[-k]` là phần tử thứ `k` từ cuối. **Bẫy:** khi tính `a[n − 4 + k]` với `n < 4`, chỉ số có thể âm và Python **không báo lỗi** mà lấy phần tử từ cuối mảng.

In [3]:
a = [3, 20, 100]
n = len(a)
for k in range(4):
    print(f"k={k}: chỉ số n-4+k = {n - 4 + k} -> a[...] = {a[n - 4 + k]}")

k=0: chỉ số n-4+k = -1 -> a[...] = 100
k=1: chỉ số n-4+k = 0 -> a[...] = 3
k=2: chỉ số n-4+k = 1 -> a[...] = 20
k=3: chỉ số n-4+k = 2 -> a[...] = 100


### 3.3 `itertools.combinations`

`combinations(range(n), r)` sinh mọi bộ `r` chỉ số tăng dần — dùng cho brute force.

In [4]:
print(list(combinations(range(4), 2)))
print("số bộ chọn 3 trong 6:", len(list(combinations(range(6), 3))))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
số bộ chọn 3 trong 6: 20


### 3.4 `heapq.nsmallest` / `heapq.nlargest`

Lấy `k` phần tử nhỏ/lớn nhất trong `O(n log k)` — với `k = 4` là tuyến tính, không cần sắp xếp cả mảng. Kết quả đã được sắp xếp (nsmallest tăng dần, nlargest giảm dần).

In [5]:
a = [9, 1, 8, 2, 7, 3, 6, 4, 5]
print(heapq.nsmallest(4, a), heapq.nlargest(4, a))

[1, 2, 3, 4] [9, 8, 7, 6]


## 4. Ý tưởng nền tảng / trực giác

**Đổi giá trị = xoá.** Khi đổi một phần tử, cách tốt nhất là đổi nó thành một giá trị **đã có** trong phần còn lại (ví dụ bằng `min` của phần còn lại) — khi đó nó không ảnh hưởng tới `max − min`. Vậy bài toán trở thành: **xoá tối đa 3 phần tử để `max − min` của phần còn lại nhỏ nhất.**

**Xoá cái gì?** Sau khi sắp xếp, phần còn lại nên là **một đoạn liên tiếp** của mảng đã sắp xếp (giữ phần tử ở giữa mà xoá phần tử "xa" là vô ích). Có `n − 3` phần tử còn lại, đoạn bắt đầu tại `k ∈ {0,1,2,3}` (xoá `k` phần tử nhỏ nhất và `3 − k` lớn nhất).

Tính tay Ví dụ 2: sắp xếp `[0, 1, 5, 10, 14]`, giữ 2 phần tử:

| k (xoá trái) | xoá phải | đoạn giữ | hiệu |
|---|---|---|---|
| 0 | 3 | [0, 1] | 1 |
| 1 | 2 | [1, 5] | 4 |
| 2 | 1 | [5, 10] | 5 |
| 3 | 0 | [10, 14] | 4 |

Nhỏ nhất = **1** ✔.

In [6]:
a = sorted([1, 5, 0, 10, 14])
n = len(a)
for k in range(4):
    keep = a[k:n - (3 - k)]
    print(f"xoá {k} trái, {3 - k} phải: giữ {keep} -> hiệu {keep[-1] - keep[0]}")

xoá 0 trái, 3 phải: giữ [0, 1] -> hiệu 1
xoá 1 trái, 2 phải: giữ [1, 5] -> hiệu 4
xoá 2 trái, 1 phải: giữ [5, 10] -> hiệu 5
xoá 3 trái, 0 phải: giữ [10, 14] -> hiệu 4


Vì sao xoá phần tử ở **giữa** là phí? Thử một phương án xoá "lộn xộn" và so với phương án đoạn liên tiếp tốt nhất:

In [7]:
a = sorted([1, 5, 0, 10, 14, 3, 8])
print("sorted:", a)
def diff_after_remove(arr, idxs):
    rest = [x for i, x in enumerate(arr) if i not in idxs]
    return max(rest) - min(rest), rest

for idxs in [{3}, {0, 3}, {0, 3, 6}, {0, 5, 6}, {4, 5, 6}, {0, 1, 2}]:
    d, rest = diff_after_remove(a, idxs)
    print(f"xoá vị trí {sorted(idxs)} -> còn {rest} -> hiệu {d}")

sorted: [0, 1, 3, 5, 8, 10, 14]
xoá vị trí [3] -> còn [0, 1, 3, 8, 10, 14] -> hiệu 14
xoá vị trí [0, 3] -> còn [1, 3, 8, 10, 14] -> hiệu 13
xoá vị trí [0, 3, 6] -> còn [1, 3, 8, 10] -> hiệu 9
xoá vị trí [0, 5, 6] -> còn [1, 3, 5, 8] -> hiệu 7
xoá vị trí [4, 5, 6] -> còn [0, 1, 3, 5] -> hiệu 5
xoá vị trí [0, 1, 2] -> còn [5, 8, 10, 14] -> hiệu 9


Quan sát: xoá vị trí 3 (ở giữa) **không làm thay đổi** `max − min`; chỉ xoá ở hai đầu mới có tác dụng. Đây chính là nền tảng của Cách 2.

## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** đúng như phép biến đổi "đổi = xoá", thử **mọi** cách xoá đúng `min(3, n−1)` phần tử (xoá nhiều hơn không bao giờ làm hiệu tăng, và giữ ít nhất 1 phần tử), tính `max − min` phần còn lại.

In [8]:
def brute(nums):
    n = len(nums)
    r = min(3, n - 1)
    best = float("inf")
    for removed in combinations(range(n), r):
        rem = set(removed)
        rest = [nums[i] for i in range(n) if i not in rem]
        best = min(best, max(rest) - min(rest))
    return best

for nums, want in examples:
    print(nums, brute(nums), "kỳ vọng", want)

[5, 3, 2, 4] 0 kỳ vọng 0
[1, 5, 0, 10, 14] 1 kỳ vọng 1
[3, 100, 20] 0 kỳ vọng 0
[6, 6, 0, 1, 1, 4, 6] 2 kỳ vọng 2


**Độ phức tạp:** `C(n, 3) ≈ n³/6` bộ, mỗi bộ `O(n)` → Time `O(n⁴)`, Space `O(n)`. Với `n = 10⁵` là bất khả thi; chỉ dùng để đối chiếu với `n ≤ 10`.

**Nút thắt:** thử cả những bộ xoá "vô lý" (xoá phần tử ở giữa). Nhận xét ở mục 4 cho thấy chỉ cần thử 4 bộ.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `a` là mảng đã sắp xếp. Phần còn lại có `max` và `min`; mọi phần tử nằm giữa `min` và `max` (theo giá trị) **không cần xoá**. Vậy tập bị xoá = "mọi phần tử nhỏ hơn min còn lại" ∪ "mọi phần tử lớn hơn max còn lại" = `k` phần tử đầu và `3 − k` phần tử cuối.

**Công thức:** với `n > 4`:

`đáp án = min_{k=0..3} ( a[n − 4 + k] − a[k] )`

(đoạn giữ là `a[k .. n−4+k]`, dài `n − 3`).

**Quy tắc:**
- *Nếu* `n ≤ 4` *thì* đổi `n − 1` phần tử thành phần tử còn lại → đáp án `0`.
- *Nếu* chỉ cần 4 phần tử nhỏ nhất và 4 lớn nhất *thì* không cần sắp xếp cả mảng → `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp + thử 4 phương án `O(n log n)`

1. Nếu `n ≤ 4` trả `0`.
2. Sắp xếp.
3. Trả `min(a[n−4+k] − a[k] for k in range(4))`.

In [9]:
def sort_four(nums):
    n = len(nums)
    if n <= 4:
        return 0
    a = sorted(nums)
    return min(a[n - 4 + k] - a[k] for k in range(4))

for nums, want in examples:
    print(nums, sort_four(nums), "kỳ vọng", want)

[5, 3, 2, 4] 0 kỳ vọng 0
[1, 5, 0, 10, 14] 1 kỳ vọng 1
[3, 100, 20] 0 kỳ vọng 0
[6, 6, 0, 1, 1, 4, 6] 2 kỳ vọng 2


In [10]:
class SolutionSort:
    def minDifference(self, nums: list[int]) -> int:
        n = len(nums)
        if n <= 4:
            return 0
        nums = sorted(nums)
        return min(nums[n - 4 + k] - nums[k] for k in range(4))

print([SolutionSort().minDifference(x) for x, _ in examples])

[0, 1, 0, 2]


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)` cho bản sao (hoặc `O(1)` phụ nếu sort tại chỗ — nhưng sửa input).

### Cách 2b — Tổng quát `k` bước: cửa sổ trượt kích thước `n − k`

Với `k` bước bất kỳ, đoạn giữ dài `w = n − k`; trượt cửa sổ trên mảng sắp xếp: `min(a[i + w − 1] − a[i])`. Bài gốc là `k = 3` (chỉ có 4 vị trí cửa sổ).

In [11]:
def min_diff_k(nums, k):
    n = len(nums)
    if n <= k + 1:
        return 0
    a = sorted(nums)
    w = n - k
    return min(a[i + w - 1] - a[i] for i in range(n - w + 1))

for nums, want in examples:
    print(nums, "k=3:", min_diff_k(nums, 3), "| k=0:", min_diff_k(nums, 0), "| k=1:", min_diff_k(nums, 1))

[5, 3, 2, 4] k=3: 0 | k=0: 3 | k=1: 2
[1, 5, 0, 10, 14] k=3: 1 | k=0: 14 | k=1: 10
[3, 100, 20] k=3: 0 | k=0: 97 | k=1: 17
[6, 6, 0, 1, 1, 4, 6] k=3: 2 | k=0: 6 | k=1: 5


### Cách 3 — `heapq.nsmallest(4)` + `heapq.nlargest(4)` `O(n)`

Công thức chỉ dùng `a[0..3]` và `a[n−4..n−1]`. Ta lấy:
- `small = nsmallest(4, nums)` → tăng dần: `small[k] = a[k]`.
- `large = sorted(nlargest(4, nums))` → tăng dần: `large[k] = a[n−4+k]`.

Đáp án `min(large[k] − small[k])`. Với `n > 4` hai tập có thể **chồng nhau** (khi `n < 8`) nhưng công thức vẫn đúng vì `large[k]` đúng là `a[n−4+k]` theo nghĩa thứ tự.

In [12]:
def heap_four(nums):
    if len(nums) <= 4:
        return 0
    small = heapq.nsmallest(4, nums)
    large = sorted(heapq.nlargest(4, nums))
    return min(large[k] - small[k] for k in range(4))

for nums, want in examples:
    print(nums, heap_four(nums), "kỳ vọng", want, "| small/large:", heapq.nsmallest(4, nums), sorted(heapq.nlargest(4, nums)))

[5, 3, 2, 4] 0 kỳ vọng 0 | small/large: [2, 3, 4, 5] [2, 3, 4, 5]
[1, 5, 0, 10, 14] 1 kỳ vọng 1 | small/large: [0, 1, 5, 10] [1, 5, 10, 14]
[3, 100, 20] 0 kỳ vọng 0 | small/large: [3, 20, 100] [3, 20, 100]
[6, 6, 0, 1, 1, 4, 6] 2 kỳ vọng 2 | small/large: [0, 1, 1, 4] [4, 6, 6, 6]


In [13]:
class SolutionHeap:
    def minDifference(self, nums: list[int]) -> int:
        if len(nums) <= 4:
            return 0
        small = heapq.nsmallest(4, nums)
        large = sorted(heapq.nlargest(4, nums))
        return min(large[k] - small[k] for k in range(4))

print([SolutionHeap().minDifference(x) for x, _ in examples])

[0, 1, 0, 2]


**Độ phức tạp:** `nsmallest/nlargest` với `k = 4` là `O(n log 4) = O(n)`, Space `O(1)` phụ.

### Cách 3b — Một lượt quét tự duy trì 4 nhỏ nhất / 4 lớn nhất (không dùng thư viện)

Minh hoạ cách "chèn vào danh sách nhỏ đã sắp xếp" — hữu ích khi phỏng vấn cấm dùng `heapq`.

In [14]:
def manual_four(nums):
    if len(nums) <= 4:
        return 0
    small, large = [], []           # small tăng dần (≤4 phần tử), large tăng dần (≤4 phần tử)
    for x in nums:
        small.append(x)
        i = len(small) - 1
        while i > 0 and small[i - 1] > small[i]:
            small[i - 1], small[i] = small[i], small[i - 1]
            i -= 1
        if len(small) > 4:
            small.pop()             # bỏ phần tử lớn nhất trong small
        large.append(x)
        i = len(large) - 1
        while i > 0 and large[i - 1] > large[i]:
            large[i - 1], large[i] = large[i], large[i - 1]
            i -= 1
        if len(large) > 4:
            large.pop(0)            # bỏ phần tử nhỏ nhất trong large
    return min(large[k] - small[k] for k in range(4))

print([manual_four(x) for x, _ in examples])

[0, 1, 0, 2]


## 8. TRACE từng bước

Trace Cách 2 với `nums = [6, 6, 0, 1, 1, 4, 6]` (`n = 7`, giữ 4):

Sắp xếp: `a = [0, 1, 1, 4, 6, 6, 6]`

| k | xoá trái | xoá phải | đoạn giữ `a[k..n−4+k]` | hiệu | best |
|---|---|---|---|---|---|
| 0 | — | 6,6,6 | [0,1,1,4] | 4 | 4 |
| 1 | 0 | 6,6 | [1,1,4,6] | 5 | 4 |
| 2 | 0,1 | 6 | [1,4,6,6] | 5 | 4 |
| 3 | 0,1,1 | — | [4,6,6,6] | 2 | 2 |

In [15]:
def trace_sort_four(nums):
    n = len(nums)
    if n <= 4:
        print("n <= 4 -> 0")
        return 0
    a = sorted(nums)
    print("sorted:", a)
    best = float("inf")
    for k in range(4):
        lo, hi = k, n - 4 + k
        diff = a[hi] - a[lo]
        best = min(best, diff)
        print(f"k={k}: xoá trái={a[:k]} xoá phải={a[hi + 1:]} giữ={a[lo:hi + 1]} hiệu={diff} best={best}")
    return best

print("kết quả:", trace_sort_four([6, 6, 0, 1, 1, 4, 6]))
print()
print("kết quả:", trace_sort_four([1, 5, 0, 10, 14]))

sorted: [0, 1, 1, 4, 6, 6, 6]
k=0: xoá trái=[] xoá phải=[6, 6, 6] giữ=[0, 1, 1, 4] hiệu=4 best=4
k=1: xoá trái=[0] xoá phải=[6, 6] giữ=[1, 1, 4, 6] hiệu=5 best=4
k=2: xoá trái=[0, 1] xoá phải=[6] giữ=[1, 4, 6, 6] hiệu=5 best=4
k=3: xoá trái=[0, 1, 1] xoá phải=[] giữ=[4, 6, 6, 6] hiệu=2 best=2
kết quả: 2

sorted: [0, 1, 5, 10, 14]
k=0: xoá trái=[] xoá phải=[5, 10, 14] giữ=[0, 1] hiệu=1 best=1
k=1: xoá trái=[0] xoá phải=[10, 14] giữ=[1, 5] hiệu=4 best=1
k=2: xoá trái=[0, 1] xoá phải=[14] giữ=[5, 10] hiệu=5 best=1
k=3: xoá trái=[0, 1, 5] xoá phải=[] giữ=[10, 14] hiệu=4 best=1
kết quả: 1


Trace Cách 3b: theo dõi `small` và `large` qua từng phần tử của Ví dụ 2:

In [16]:
def trace_manual(nums):
    small, large = [], []
    for x in nums:
        small = sorted(small + [x])[:4]
        large = sorted(large + [x])[-4:]
        print(f"x={x:3d} small={small} large={large}")
    if len(nums) > 4:
        print("hiệu theo k:", [large[k] - small[k] for k in range(4)])

trace_manual([1, 5, 0, 10, 14, 7, -3])

x=  1 small=[1] large=[1]
x=  5 small=[1, 5] large=[1, 5]
x=  0 small=[0, 1, 5] large=[0, 1, 5]
x= 10 small=[0, 1, 5, 10] large=[0, 1, 5, 10]
x= 14 small=[0, 1, 5, 10] large=[1, 5, 10, 14]
x=  7 small=[0, 1, 5, 7] large=[5, 7, 10, 14]
x= -3 small=[-3, 0, 1, 5] large=[5, 7, 10, 14]
hiệu theo k: [8, 7, 9, 9]


## 9. Code nộp LeetCode

Bản chính: sắp xếp + 4 phương án (gọn, dễ đọc). Bản `O(n)` là `SolutionHeap`.

In [17]:
class Solution:
    def minDifference(self, nums: list[int]) -> int:
        n = len(nums)
        if n <= 4:
            return 0
        nums = sorted(nums)
        return min(nums[n - 4 + k] - nums[k] for k in range(4))

for nums, want in examples:
    assert Solution().minDifference(nums) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Bước 1 (đổi = xoá):** Gọi `R` là tập phần tử không bị đổi (`|R| ≥ n − 3`). Sau khi đổi, `max − min ≥ max(R) − min(R)`. Và đạt được dấu bằng bằng cách đổi mọi phần tử bị chọn thành `min(R)`. Vậy đáp án = `min` trên mọi `R` có `|R| ≥ n − 3` của `max(R) − min(R)`. Nếu `n ≤ 4` chọn `|R| = 1` → 0.

**Bước 2 (R là đoạn liên tiếp trong mảng sắp xếp):** Cho `R` tối ưu với `min(R) = a[i]`, `max(R) = a[j]`. Thay `R` bằng `a[i..j]` (thêm mọi phần tử nằm giữa) không làm thay đổi `max − min` và chỉ làm `R` lớn hơn (vẫn hợp lệ). Thu nhỏ về đúng `n − 3` phần tử bằng cách bỏ bớt ở hai đầu chỉ làm hiệu giảm hoặc giữ nguyên. Vậy tồn tại `R` tối ưu là đoạn `a[k .. k + n − 4]` với `k ∈ {0,1,2,3}`.

**Bước 3:** thử cả 4 giá trị `k` → lấy min → đúng. ∎

Kiểm chứng bằng thực nghiệm hai bước của chứng minh: (a) với mọi phương án xoá 3 phần tử, đổi chúng thành `min` của phần còn lại cho đúng hiệu của phần còn lại; (b) phương án tối ưu của brute force luôn bằng một trong 4 "đoạn liên tiếp".

In [18]:
rng_proof = random.Random(10)
for _ in range(300):
    nums = [rng_proof.randint(-30, 30) for _ in range(rng_proof.randint(5, 8))]
    n = len(nums)
    for rem in combinations(range(n), 3):
        rest = [nums[i] for i in range(n) if i not in rem]
        changed = [min(rest) if i in rem else nums[i] for i in range(n)]
        assert max(changed) - min(changed) == max(rest) - min(rest)      # (a)
    a = sorted(nums)
    windows = [a[n - 4 + k] - a[k] for k in range(4)]
    assert brute(nums) == min(windows)                                    # (b)
print("Hai bước chứng minh được kiểm chứng trên 300 mảng ngẫu nhiên")

Hai bước chứng minh được kiểm chứng trên 300 mảng ngẫu nhiên


## 11. Phân tích Time / Space

Gọi `n = len(nums) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(C(n,3)·n) = O(n⁴)` | `O(n)` |
| 2. Sort + 4 phương án | `O(n log n)` | `O(n)` (bản sao) |
| 2b. Tổng quát `k` | `O(n log n)` | `O(n)` |
| 3. nsmallest/nlargest | `O(n)` | `O(1)` |

Không có khác biệt average/worst: sắp xếp Timsort luôn `O(n log n)` worst.

## 12. Bộ test

Ví dụ đề, edge case (`n = 1..5`, phần tử trùng, số âm lớn), test ngẫu nhiên đối chiếu brute force (seed cố định) và test lớn `n = 10⁵`.

In [19]:
METHODS = {
    "sort": lambda a: SolutionSort().minDifference(a),
    "heap": lambda a: SolutionHeap().minDifference(a),
    "manual": manual_four,
    "k=3": lambda a: min_diff_k(a, 3),
    "final": lambda a: Solution().minDifference(a),
}
edge = [
    ([7], 0), ([1, 2], 0), ([1, 10, 100, 1000], 0),
    ([1, 10, 100, 1000, 10000], 9),
    ([-10**9, 10**9, 0, 0, 0, 0], 0),
    ([5, 5, 5, 5, 5, 5], 0),
    ([0, 100, 101, 102, 103, 200], 2),
]
for nums, want in examples + edge:
    for name, f in METHODS.items():
        got = f(list(nums))
        assert got == want, (name, nums, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [20]:
rng = random.Random(1509)
for _ in range(600):
    n = rng.randint(1, 9)
    span = rng.choice([3, 20, 10**9])
    nums = [rng.randint(-span, span) for _ in range(n)]
    ref = brute(nums)
    for name, f in METHODS.items():
        assert f(list(nums)) == ref, (name, nums)
print("600 test ngẫu nhiên: mọi cách khớp brute")

600 test ngẫu nhiên: mọi cách khớp brute


In [21]:
big = [random.Random(5).randint(-10**9, 10**9) for _ in range(10**5)]
for name in ["sort", "heap", "manual"]:
    t = time.perf_counter()
    r = METHODS[name](big)
    print(f"{name:6s}: {r} ({time.perf_counter() - t:.3f}s)")
assert METHODS["sort"](big) == METHODS["heap"](big) == METHODS["manual"](big)

sort  : 0 (0.002s)
heap  : 0 (0.004s)
manual: 0 (0.024s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Luôn xoá 3 phần tử lớn nhất (hoặc 3 nhỏ nhất)

Chỉ là 2 trong 4 phương án.

In [22]:
def wrong_only_right(nums):
    if len(nums) <= 4:
        return 0
    a = sorted(nums)
    return a[-4] - a[0]                      # SAI: chỉ xoá 3 phần tử lớn nhất

def wrong_only_left(nums):
    if len(nums) <= 4:
        return 0
    a = sorted(nums)
    return a[-1] - a[3]                      # SAI: chỉ xoá 3 phần tử nhỏ nhất

bad = [0, 100, 101, 102, 103, 200]
print("xoá 3 lớn :", wrong_only_right(bad), "| xoá 3 nhỏ:", wrong_only_left(bad),
      "| đúng:", Solution().minDifference(bad))

xoá 3 lớn : 101 | xoá 3 nhỏ: 98 | đúng: 2


### Lỗi 2 — Quên trường hợp `n ≤ 4` → chỉ số âm / vượt biên

Với `n < 4`, `a[n − 4 + k]` là chỉ số **âm** (Python lặng lẽ lấy từ cuối mảng → hiệu có thể âm), còn `a[k]` có thể **vượt biên** → `IndexError`. Bản sai dưới đây chỉ lặp `k < n` để thấy kết quả âm, và bản thứ hai cho thấy exception (ta bắt lại để in ra).

In [23]:
def wrong_small_n(nums):
    n = len(nums)
    a = sorted(nums)                          # SAI: thiếu if n <= 4: return 0
    return min(a[n - 4 + k] - a[k] for k in range(min(4, n)))

def wrong_small_n2(nums):
    n = len(nums)
    a = sorted(nums)                          # SAI: thiếu kiểm tra n
    return min(a[n - 4 + k] - a[k] for k in range(4))

for bad in [[3, 100, 20], [1, 2]]:
    print(bad, "| sai (bản 1):", wrong_small_n(bad), "| đúng:", Solution().minDifference(bad))
    try:
        print("   sai (bản 2):", wrong_small_n2(bad))
    except IndexError as e:
        print("   sai (bản 2): IndexError ->", e)

[3, 100, 20] | sai (bản 1): -80 | đúng: 0
   sai (bản 2): IndexError -> list index out of range
[1, 2] | sai (bản 1): 0 | đúng: 0
   sai (bản 2): IndexError -> list index out of range


### Lỗi 3 — Tham lam cục bộ: mỗi bước xoá đầu có "khoảng hở" lớn hơn

Nghe hợp lý (xoá phần tử "lạc loài" nhất) nhưng không nhìn trước được các bước sau.

In [24]:
def wrong_greedy(nums):
    a = sorted(nums)
    for _ in range(3):
        if len(a) <= 1:
            return 0
        if a[1] - a[0] >= a[-1] - a[-2]:     # SAI: chọn cục bộ
            a.pop(0)
        else:
            a.pop()
    return a[-1] - a[0]

bad = [1, 3, 3, 4, 1, 1]
print("sai:", wrong_greedy(bad), "| đúng:", Solution().minDifference(bad), "(xoá 3, 3, 4 -> [1,1,1])")

sai: 2 | đúng: 0 (xoá 3, 3, 4 -> [1,1,1])


### Lỗi 4 — Dùng `nlargest(4)` mà quên đảo về thứ tự tăng

`nlargest` trả theo thứ tự **giảm dần**; ghép chỉ số `k` với `nsmallest` (tăng dần) sẽ sai.

In [25]:
def wrong_order(nums):
    if len(nums) <= 4:
        return 0
    small = heapq.nsmallest(4, nums)
    large = heapq.nlargest(4, nums)           # SAI: giảm dần, phải sorted(...)
    return min(large[k] - small[k] for k in range(4))

bad = [1, 5, 0, 10, 14]
print("sai:", wrong_order(bad), "| đúng:", Solution().minDifference(bad))
bad2 = [0, 100, 101, 102, 103, 200]
print("sai:", wrong_order(bad2), "| đúng:", Solution().minDifference(bad2))

sai: -9 | đúng: 1
sai: -1 | đúng: 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đổi tối đa k phần tử thành giá trị tuỳ ý, tối thiểu hoá max − min"** → đổi = xoá → giữ một **đoạn liên tiếp** dài `n − k` trong mảng sắp xếp.
- `k` nhỏ cố định (3) → chỉ có `k + 1` phương án, có thể dùng `nsmallest/nlargest` để đạt `O(n)`.
- Luôn kiểm tra trước `n ≤ k + 1` → đáp án 0.
- Khi nghi ngờ một tham lam, viết brute force nhỏ và chạy random để tìm phản ví dụ (như Lỗi 3).
- Dấu hiệu "chỉ phụ thuộc vài phần tử nhỏ nhất/lớn nhất" → không cần sắp xếp toàn bộ.

## 15. Biến thể của bài

1. **k bước bất kỳ** → cửa sổ trượt (Cách 2b).
2. **Bài ngược:** số bước ít nhất để `max − min ≤ d` → `n − (đoạn dài nhất có a[j] − a[i] ≤ d)` (hai con trỏ).
3. **LC 1658 / 2009 – Minimum Number of Operations to Make Array Continuous:** cùng ý tưởng "giữ một đoạn".
4. **LC 908 / 910 – Smallest Range I / II:** mỗi phần tử được cộng/trừ `k`.
5. **LC 2567 – Minimum Score by Changing Two Elements:** phiên bản 2 bước (3 phương án).

In [26]:
# Biến thể 5 (2 bước): tổng quát hoá bằng min_diff_k
print([min_diff_k(x, 2) for x, _ in examples])
print("LC 2567 ví dụ [1,4,7,8,5] với 2 bước:", min_diff_k([1, 4, 7, 8, 5], 2))

[1, 5, 0, 5]
LC 2567 ví dụ [1,4,7,8,5] với 2 bước: 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Thử mọi bộ 3 phần tử để xoá | `O(n⁴)` | `O(n)` | Đối chiếu, `n ≤ 10` |
| 2. Sort + 4 phương án | Giữ đoạn `a[k..n−4+k]` | `O(n log n)` | `O(n)` | Mặc định, gọn nhất |
| 2b. Cửa sổ `n − k` | Tổng quát cho mọi `k` | `O(n log n)` | `O(n)` | Khi `k` thay đổi |
| 3. nsmallest/nlargest | Chỉ cần 4 nhỏ + 4 lớn | `O(n)` | `O(1)` | Tối ưu tiệm cận |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().minDifference([1, 5, 6, 14, 15])` trả về bao nhiêu?

### Bài 2 (tính tay)
Với `nums = [20, 75, 81, 82, 95]`, lập bảng 4 phương án và tìm đáp án.

### Bài 3 (điền TODO)
Viết `min_moves_within(nums, d)`: số bước **ít nhất** để `max − min ≤ d` (mỗi bước đổi một phần tử thành giá trị tuỳ ý).

### Bài 4 (tự viết lại)
Viết lại Cách 3 **không dùng** `heapq` nhưng vẫn `O(n)`: chỉ dùng 8 biến/hai list độ dài 4.

### Bài 5 (mini challenge)
Ngoài hiệu nhỏ nhất, trả về **mảng sau khi đổi** (một mảng hợp lệ bất kỳ đạt hiệu đó, đổi tối đa 3 phần tử): `min_difference_with_array(nums)`.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def min_moves_within(nums, d):
    # TODO: sort; hai con trỏ tìm đoạn dài nhất có a[j] - a[i] <= d; đáp án n - độ dài
    pass

# Bài 4 — TODO
def min_difference_no_heap(nums):
    # TODO: duy trì small (4 nhỏ nhất, tăng dần) và large (4 lớn nhất, tăng dần) trong 1 lượt
    pass

# Bài 5 — TODO
def min_difference_with_array(nums):
    # TODO: tìm k tốt nhất; đổi k phần tử nhỏ nhất và 3-k lớn nhất thành a[k]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `n = 5` → giữ 2 phần tử. (2) Bốn cặp liên tiếp `(1,5),(5,6),(6,14),(14,15)`. (3) Min hiệu = 1.

**Bài 2.** (1) Đã sắp xếp sẵn. (2) Giữ 2: `(20,75),(75,81),(81,82),(82,95)`. (3) Đáp án 1.

**Bài 3.** (1) Phần tử giữ lại tạo thành một đoạn liên tiếp trong mảng sắp xếp. (2) Hai con trỏ: với mỗi `j`, tiến `i` khi `a[j] − a[i] > d`. (3) Đáp án `n − max(j − i + 1)`.

**Bài 4.** (1) Dùng chèn có sắp xếp trên list độ dài ≤ 4 (như `manual_four`). (2) Mỗi phần tử tốn `O(4)`. (3) Đừng quên `n ≤ 4`.

**Bài 5.** (1) Tìm `k` đạt min. (2) Chỉ số cần đổi: `k` chỉ số nhỏ nhất và `3 − k` chỉ số lớn nhất (theo thứ tự sắp xếp, dùng `sorted(range(n), key=...)`). (3) Gán chúng bằng `a[k]` rồi kiểm tra `max − min` và số phần tử thay đổi ≤ 3.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
r1 = Solution().minDifference([1, 5, 6, 14, 15])
a = [20, 75, 81, 82, 95]
table = [(k, a[k:len(a) - (3 - k)]) for k in range(4)]
r2 = Solution().minDifference(a)
print("Bài 1:", r1, "| Bài 2:", table, "->", r2)
assert r1 == 1 and r2 == 1

Bài 1: 1 | Bài 2: [(0, [20, 75]), (1, [75, 81]), (2, [81, 82]), (3, [82, 95])] -> 1


In [29]:
# Bài 3
def min_moves_within_sol(nums, d):
    a = sorted(nums)
    best = i = 0
    for j in range(len(a)):
        while a[j] - a[i] > d:
            i += 1
        best = max(best, j - i + 1)
    return len(a) - best

def brute_moves(nums, d):
    n = len(nums)
    for r in range(n):
        for rem in combinations(range(n), r):
            rest = [nums[i] for i in range(n) if i not in rem]
            if max(rest) - min(rest) <= d:
                return r
    return n - 1

# Bài 4
def min_difference_no_heap_sol(nums):
    return manual_four(nums)

rng = random.Random(3)
for _ in range(300):
    nums = [rng.randint(-20, 20) for _ in range(rng.randint(1, 8))]
    d = rng.randint(0, 15)
    assert min_moves_within_sol(nums, d) == brute_moves(nums, d)
    assert min_difference_no_heap_sol(nums) == brute(nums)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [30]:
# Bài 5
def min_difference_with_array_sol(nums):
    n = len(nums)
    if n <= 4:
        return 0, [nums[0]] * n
    order = sorted(range(n), key=lambda i: nums[i])
    a = [nums[i] for i in order]
    k = min(range(4), key=lambda k: a[n - 4 + k] - a[k])
    res = nums[:]
    for idx in order[:k] + order[n - (3 - k):]:
        res[idx] = a[k]
    return a[n - 4 + k] - a[k], res

for _ in range(300):
    nums = [rng.randint(-50, 50) for _ in range(rng.randint(1, 9))]
    diff, arr = min_difference_with_array_sol(nums)
    assert diff == Solution().minDifference(nums)
    assert max(arr) - min(arr) == diff
    assert sum(x != y for x, y in zip(nums, arr)) <= 3
print(min_difference_with_array_sol([1, 5, 0, 10, 14]))
print("Đáp án Bài 1–5: mọi assert đều qua")

(1, [1, 0, 0, 0, 0])
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đổi tối đa k phần tử, tối thiểu max − min":**

```python
n = len(nums)
if n <= k + 1:
    return 0
a = sorted(nums)
w = n - k                                   # số phần tử giữ lại
return min(a[i + w - 1] - a[i] for i in range(k + 1))
```

**Checklist trước khi submit:**
- [ ] Xử lý `n ≤ 4` → `0` (tránh chỉ số âm).
- [ ] Thử đủ 4 phương án (xoá `k` trái, `3 − k` phải), không chỉ một phía.
- [ ] Không dùng tham lam cục bộ từng bước.
- [ ] Nếu dùng `nlargest`, nhớ `sorted(...)` về tăng dần.
- [ ] Đúng chữ ký: `minDifference(self, nums: list[int]) -> int`.